# Topic 3 — Climate & Energy: Combined Data Cleaning

**Part I:** EV charging datasets (sections 1–14).

**Part II:** Grid and weather datasets, hourly merge, and daily averages (sections 15–22).

**Part III:** Grid and weather datasets, hourly merge, and daily averages (sections 23–27).

Run cells in order. Open the notebook from the project root or the Topic 3 `data` folder. Generated CSV files are written to the Topic 3 data folder.


# 1. Import Libraries and Load the Datasets

Pandas and NumPy are used to load, inspect, clean, and integrate the datasets.

Three datasets are loaded for this stage:

- **Charging Sessions:** Contains individual EV charging sessions, including the vehicle, station, start time, end time, charging type, and energy delivered.
- **Vehicles:** Contains information about the 10,000 EVs, including vehicle identifiers, battery capacity, registration month, and gasoline vehicle information.
- **Public Stations:** Contains information about public charging stations and their identifiers.

Identifiers are loaded as strings to preserve their original formatting. 

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Works when launched from the project root or the Topic 3 data folder.
project_data = Path("Topic3_Climate_Energy_The_Green_Tech_Carbon_Spike/data")
if not project_data.is_dir():
    project_data = Path(".")
if not (project_data / "charging_sessions.csv.gz").exists():
    raise FileNotFoundError("Open this notebook from the project root or Topic 3 data folder.")

# Load datasets while preserving IDs as text
charging = pd.read_csv(project_data / "charging_sessions.csv.gz", dtype={
    "session_id": "string",
    "vehicle_id": "string",
    "station_id": "string"
})

vehicles = pd.read_csv(project_data / "vehicles.csv", dtype={
    "vehicle_id": "string",
    "home_zip": "string"
})

stations = pd.read_csv(project_data / "stations.csv", dtype={
    "station_id": "string",
    "zip": "string"
})

print("Datasets loaded successfully!")

Datasets loaded successfully!


# 2. Record Original Row Counts

Before making any changes, the original number of rows in each dataset is recorded.

These counts provide a baseline for measuring how many records remain after cleaning.

The original and final row counts will be compared at the end of the notebook. Each cleaning operation is also recorded separately so that removed records and the reasons for their removal can be reported.

In [3]:
original_counts = {
    "Charging Sessions": len(charging),
    "Vehicles": len(vehicles),
    "Public Stations": len(stations)
}

for name, count in original_counts.items():
    print(f"{name}: {count:,} rows")

Charging Sessions: 2,927,187 rows
Vehicles: 10,000 rows
Public Stations: 239 rows


# 3. Inspect the Original Datasets

The inspection reports:
- Dataset dimensions (rows and columns).
- Missing values in each column.
- Completely duplicated rows.
- Data types for each column.

This initial inspection helps identify problems that may affect later analysis, including missing energy measurements, duplicate records, incorrect data types, and incomplete identifiers.

The results are used to determine which cleaning steps are necessary.

In [4]:
for name, df in {
    "Charging Sessions": charging,
    "Vehicles": vehicles,
    "Public Stations": stations
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)


Charging Sessions
Shape: (2927187, 9)

Missing values:
session_id            0
vehicle_id            0
station_id            0
port_type             0
connector             0
start_time            0
end_time              0
kwh               44045
home_or_public        0
dtype: int64

Duplicate rows: 14563

Data types:
session_id         string
vehicle_id         string
station_id         string
port_type             str
connector             str
start_time            str
end_time              str
kwh               float64
home_or_public        str
dtype: object

Vehicles
Shape: (10000, 10)

Missing values:
vehicle_id              0
make                    0
model                   0
trim                    0
model_year              0
battery_kwh             0
rated_kwh_per_100mi     0
registration_month      0
replaced_gas_car_mpg    0
home_zip                0
dtype: int64

Duplicate rows: 0

Data types:
vehicle_id               string
make                        str
model           

# 4. Create Cleaning and Data-Quality Logs

Two logs are created to document the cleaning process.

**Cleaning log:** Records operations that remove rows. For each operation, it records:
- Dataset name.
- Cleaning operation.
- Number of rows before cleaning.
- Number of rows after cleaning.
- Number of rows removed.
- Reason for the operation.

**Data-quality log:** Records invalid values or conversions that do not necessarily remove entire rows. For example, a value that cannot be converted into a number is changed to a missing value (`NaN`) and recorded.

Keeping these logs separate makes it possible to distinguish between records removed from the analysis and individual values that require further investigation.

In [5]:
cleaning_log = []

def log_cleaning(dataset, step, before, after, reason):
    cleaning_log.append({
        "Dataset": dataset,
        "Cleaning Step": step,
        "Rows Before": before,
        "Rows After": after,
        "Rows Removed": before - after,
        "Reason": reason
    })

def show_cleaning_log():
    return pd.DataFrame(cleaning_log)

In [6]:
# Track data-quality issues that do not necessarily remove rows
quality_log = []

def log_quality(dataset, column, affected, action, reason):
    quality_log.append({
        "Dataset": dataset,
        "Column": column,
        "Affected Values": affected,
        "Action": action,
        "Reason": reason
    })

def show_quality_log():
    return pd.DataFrame(quality_log)

# 5. Clean the Vehicle Dataset

The vehicle dataset is cleaned first because vehicle identifiers are needed to verify and connect charging sessions.

The following operations are performed:

1. **Standardize vehicle IDs:** Remove extra spaces and convert IDs to uppercase so that formatting differences do not prevent matches.
2. **Remove missing vehicle IDs:** Records without an ID cannot be reliably connected to charging sessions.
3. **Remove duplicate vehicle IDs:** Each vehicle should appear only once in the vehicle dataset. When duplicate IDs occur, the first record is retained.
4. **Convert numerical columns:** Battery capacity, rated energy consumption, and replaced gasoline vehicle MPG are converted to numeric values. Values that cannot be converted become `NaN` and are recorded in the data-quality log.
5. **Convert registration month:** Registration dates are converted into datetime values so vehicles can be analyzed according to when they entered the fleet. Invalid date values become `NaT` and are recorded.

The vehicle dataset is not reduced because of missing non-identifying vehicle information. Such values remain available for later investigation.

In [7]:
vehicles_clean = vehicles.copy()

# Remove extra spaces and standardize IDs
vehicles_clean["vehicle_id"] = vehicles_clean["vehicle_id"].str.strip().str.upper()

# Remove records without vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.dropna(subset=["vehicle_id"])
log_cleaning(
    "Vehicles",
    "Remove missing vehicle IDs",
    before,
    len(vehicles_clean),
    "A vehicle must have an ID to connect it to charging sessions."
)

# Remove duplicate vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.drop_duplicates(
    subset=["vehicle_id"],
    keep="first"
)
log_cleaning(
    "Vehicles",
    "Remove duplicate vehicle IDs",
    before,
    len(vehicles_clean),
    "Each vehicle should appear only once."
)

numeric_columns = [
    "battery_kwh",
    "rated_kwh_per_100mi",
    "replaced_gas_car_mpg"
]

for column in numeric_columns:

    before_invalid = vehicles_clean[column].isna().sum()

    original_values = vehicles_clean[column].copy()

    vehicles_clean[column] = pd.to_numeric(
        vehicles_clean[column],
        errors="coerce"
    )

    newly_invalid = (
        original_values.notna() &
        vehicles_clean[column].isna()
    ).sum()

    log_quality(
        "Vehicles",
        column,
        newly_invalid,
        "Converted invalid numerical values to NaN",
        "The column must contain numeric values for calculations."
    )

original_registration = vehicles_clean["registration_month"].copy()

vehicles_clean["registration_month"] = pd.to_datetime(
    vehicles_clean["registration_month"],
    format="%Y-%m",
    errors="coerce"
)

invalid_registration = (
    original_registration.notna() &
    vehicles_clean["registration_month"].isna()
).sum()

log_quality(
    "Vehicles",
    "registration_month",
    invalid_registration,
    "Converted invalid dates to NaT",
    "Registration dates are needed to identify when vehicles entered the fleet."
)

print("Clean vehicle records:", len(vehicles_clean))
print("Unique vehicle IDs:", vehicles_clean["vehicle_id"].nunique())

Clean vehicle records: 10000
Unique vehicle IDs: 10000


# 6. Clean the Public Charging Station Dataset

The public station dataset is prepared so that public charging sessions can be matched to known stations.

The following operations are performed:

1. **Standardize station IDs:** Remove extra spaces and convert station IDs to uppercase.
2. **Remove missing station IDs:** A station without an identifier cannot be matched to a charging session.
3. **Remove duplicate station IDs:** Each public station should appear only once. When duplicate IDs occur, the first record is retained.
4. **Convert opening dates:** The station opening date is converted to a datetime value. Invalid dates become `NaT`.

Home charging stations are not expected to appear in this dataset because it contains public stations only. Home charging stations are checked separately using the vehicle identifier.

In [8]:
stations_clean = stations.copy()

# Standardize station IDs
stations_clean["station_id"] = (
    stations_clean["station_id"].str.strip().str.upper()
)

# Remove missing station IDs
before = len(stations_clean)
stations_clean = stations_clean.dropna(subset=["station_id"])

log_cleaning(
    "Stations",
    "Remove missing station IDs",
    before,
    len(stations_clean),
    "A public station must have an ID to match charging sessions."
)

# Remove duplicate station IDs
before = len(stations_clean)
stations_clean = stations_clean.drop_duplicates(
    subset=["station_id"],
    keep="first"
)

log_cleaning(
    "Stations",
    "Remove duplicate station IDs",
    before,
    len(stations_clean),
    "Each public charging station should appear once."
)

# Convert station opening date
stations_clean["open_date"] = pd.to_datetime(
    stations_clean["open_date"],
    errors="coerce"
)

print("Clean public stations:", len(stations_clean))
print("Unique station IDs:", stations_clean["station_id"].nunique())

Clean public stations: 239
Unique station IDs: 239


# 7. Clean the Charging Sessions Dataset

The charging sessions dataset contains the individual charging events used in the project's analysis. It is therefore important to preserve valid sessions while identifying incomplete or inconsistent records.

The following operations are performed:

1. **Standardize text fields:** Remove leading and trailing spaces from identifiers and descriptive columns.
2. **Standardize identifiers:** Convert session, vehicle, and station IDs to uppercase.
3. **Standardize charging location:** Convert the home/public field to lowercase so that the two categories can be consistently identified.
4. **Standardize charger information:** Convert port types to lowercase and connector types to uppercase.
5. **Convert energy measurements:** Convert `kwh` to a numeric value. Values that cannot be converted become `NaN` and are recorded in the data-quality log.
6. **Convert timestamps:** Convert session start and end times into datetime values. Invalid timestamps become `NaT` and are recorded.

The charging timestamps are recorded as local clock times without an explicit UTC offset. This creates a potential daylight-saving-time ambiguity during the fall clock change, when the 1:00 AM hour occurs twice.

For this reason, duration validation is postponed until the daylight-saving-time issue can be investigated. This avoids incorrectly removing potentially valid sessions.

In [9]:
charging_clean = charging.copy()

# Standardize text columns
text_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "port_type",
    "connector",
    "home_or_public"
]

for column in text_columns:
    charging_clean[column] = charging_clean[column].str.strip()

# Standardize identifiers
charging_clean["session_id"] = charging_clean["session_id"].str.upper()
charging_clean["vehicle_id"] = charging_clean["vehicle_id"].str.upper()
charging_clean["station_id"] = charging_clean["station_id"].str.upper()

# Standardize charging location
charging_clean["home_or_public"] = (
    charging_clean["home_or_public"].str.lower()
)

# Standardize charger types
charging_clean["port_type"] = (
    charging_clean["port_type"].str.lower()
)

charging_clean["connector"] = (
    charging_clean["connector"].str.upper()
)

original_kwh = charging_clean["kwh"].copy()

charging_clean["kwh"] = pd.to_numeric(
    charging_clean["kwh"],
    errors="coerce"
)

invalid_kwh = (
    original_kwh.notna() &
    charging_clean["kwh"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "kwh",
    invalid_kwh,
    "Converted invalid numerical values to NaN",
    "Energy consumption must be numeric to calculate charging emissions."
)

# Convert start timestamps
original_start = charging_clean["start_time"].copy()

charging_clean["start_time"] = pd.to_datetime(
    charging_clean["start_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

invalid_start = (
    original_start.notna() &
    charging_clean["start_time"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "start_time",
    invalid_start,
    "Converted invalid timestamps to NaT",
    "Valid timestamps are required to determine when charging occurred."
)

# Convert end timestamps
original_end = charging_clean["end_time"].copy()

charging_clean["end_time"] = pd.to_datetime(
    charging_clean["end_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

invalid_end = (
    original_end.notna() &
    charging_clean["end_time"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "end_time",
    invalid_end,
    "Converted invalid timestamps to NaT",
    "Valid timestamps are required to calculate charging duration."
)

print("Charging sessions inspected:", len(charging_clean))

Charging sessions inspected: 2927187


# 7.1 Preserve and Remove Sessions with Missing Essential Information

Before removing incomplete charging sessions, records with missing energy measurements are saved separately as `sessions_missing_energy.csv`.

These records are preserved for investigation because missing energy measurements cannot be used to calculate charging emissions. They are not automatically assigned an energy value of zero.

The notebook then reports the number of missing values in each required column.

The following fields are considered essential:
- Session ID.
- Vehicle ID.
- Station ID.
- Start time.
- End time.
- Energy delivered (`kwh`).
- Charging location (`home_or_public`).

Records missing any of these required values are removed from the main analysis dataset and the number removed is recorded.

This step ensures that the remaining sessions have the minimum information necessary for matching, time-based analysis, and energy calculations.

In [10]:
required_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "start_time",
    "end_time",
    "kwh",
    "home_or_public"
]

before = len(charging_clean)

missing_energy_sessions = charging_clean[
    charging_clean["kwh"].isna()
].copy()

missing_energy_sessions.to_csv(
    "sessions_missing_energy.csv",
    index=False
)

print(
    "Sessions saved for missing-energy investigation:",
    len(missing_energy_sessions)
)

print("MISSING REQUIRED VALUES BEFORE REMOVAL")
print("--------------------------------------")

for column in required_columns:
    print(f"{column}: {charging_clean[column].isna().sum():,}")
charging_clean = charging_clean.dropna(
    subset=required_columns
)

log_cleaning(
    "Charging Sessions",
    "Remove missing essential values",
    before,
    len(charging_clean),
    "Sessions need identifiers, timestamps, electricity consumption, and charging location."
)

Sessions saved for missing-energy investigation: 44045
MISSING REQUIRED VALUES BEFORE REMOVAL
--------------------------------------
session_id: 0
vehicle_id: 0
station_id: 0
start_time: 0
end_time: 0
kwh: 44,045
home_or_public: 0


# 7.2 Remove Duplicate Charging Sessions

Charging sessions are checked for duplicate session IDs.

Each session should represent one unique charging event. If the same session ID appears more than once, the notebook retains the first occurrence and removes subsequent occurrences.

This prevents the same charging event from being counted multiple times when calculating total energy consumption and emissions.

In [11]:
before = len(charging_clean)

charging_clean = charging_clean.drop_duplicates(
    subset=["session_id"],
    keep="first"
)

log_cleaning(
    "Charging Sessions",
    "Remove duplicate session IDs",
    before,
    len(charging_clean),
    "Each charging session should appear only once."
)

# 7.3 Remove Invalid Energy and Charging Location Values

Charging sessions are checked for two additional problems:

1. **Invalid energy:** Sessions must have a positive energy value (`kwh > 0`).
2. **Invalid charging location:** The location must be identified as either `home` or `public`.

Session duration is intentionally not validated at this stage. Because the timestamps do not contain timezone offsets, a session crossing the repeated daylight-saving-time hour could appear to have a negative duration when interpreted as ordinary local timestamps. Duration validation will be addressed separately during timestamp and daylight-saving-time analysis.

In [12]:
# Remove invalid energy and location values only.
# Duration validation will happen after DST ambiguity is investigated.

before = len(charging_clean)

charging_clean = charging_clean[
    (charging_clean["kwh"] > 0) &
    (charging_clean["home_or_public"].isin(["home", "public"]))
].copy()

log_cleaning(
    "Charging Sessions",
    "Remove invalid energy and location records",
    before,
    len(charging_clean),
    "Sessions must have positive energy and a recognized charging location. Duration validation is deferred until DST handling."
)

# 8. Match Charging Sessions to Vehicles

Each charging session must be associated with one of the vehicles in the vehicle dataset.

The vehicle IDs from the cleaned vehicle dataset are compared with the vehicle IDs in the cleaned charging sessions dataset.

Two groups are identified:

- **Matched sessions:** The vehicle ID exists in the vehicle dataset.
- **Unmatched sessions:** The vehicle ID does not exist in the vehicle dataset.

The number of matched and unmatched sessions is reported.

Sessions without a matching vehicle are removed from the main analysis dataset because they cannot be reliably associated with the EV fleet being studied. The number removed and the reason are recorded.

In [13]:
# Identify sessions with no matching vehicle
vehicle_ids = set(vehicles_clean["vehicle_id"])

charging_clean["vehicle_match"] = (
    charging_clean["vehicle_id"].isin(vehicle_ids)
)

unmatched_vehicles = charging_clean[
    ~charging_clean["vehicle_match"]
].copy()

print("Sessions with matching vehicles:",
      charging_clean["vehicle_match"].sum())

print("Sessions without matching vehicles:",
      len(unmatched_vehicles))

Sessions with matching vehicles: 2868765
Sessions without matching vehicles: 0


In [14]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["vehicle_match"]
].copy()

charging_clean = charging_clean.drop(columns=["vehicle_match"])

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching vehicles",
    before,
    len(charging_clean),
    "Every analyzed charging session must belong to one of the 10,000 mandate vehicles."
)

# 9. Match Charging Sessions to Stations

Charging sessions are matched to stations using different rules for home and public charging.

**Public charging:**
- The session's station ID must exist in the cleaned public station dataset.

**Home charging:**
- Home station IDs follow the expected format `H-` followed by the vehicle ID.
- The session's station ID must match the expected home station ID for that vehicle.

Sessions without a valid station relationship are removed from the main analysis dataset. The number removed and the reason are recorded.


In [15]:
public_station_ids = set(stations_clean["station_id"])

# Check public station matches
charging_clean["station_match"] = np.where(
    charging_clean["home_or_public"] == "public",
    charging_clean["station_id"].isin(public_station_ids),

    # Home station must match its vehicle ID
    charging_clean["station_id"] ==
    "H-" + charging_clean["vehicle_id"]
)

# Identify unmatched stations
unmatched_stations = charging_clean[
    ~charging_clean["station_match"]
].copy()

print("Sessions with matching stations:",
      charging_clean["station_match"].sum())

print("Sessions without matching stations:",
      len(unmatched_stations))

Sessions with matching stations: 2868765
Sessions without matching stations: 0


In [16]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["station_match"]
].copy()

charging_clean = charging_clean.drop(
    columns=["station_match"]
)

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching stations",
    before,
    len(charging_clean),
    "Public sessions must match stations.csv; home sessions must match their vehicle's H- station ID."
)

# 10. Check Charging Energy Against Vehicle Battery Capacity

The cleaned vehicle dataset is merged with the charging sessions dataset to add each vehicle's usable battery capacity.

Each session's delivered energy is then compared with the vehicle's battery capacity.

Sessions with delivered energy greater than the listed battery capacity are flagged as potential data-quality issues and saved to `battery_capacity_issues.csv`.

This data is not automatically removed. They are preserved for further investigation because an unusually large session may indicate an error, an unusual charging event, or another issue.

This check helps identify potentially unrealistic energy measurements before calculating total EV charging energy and emissions.

In [17]:
# Add vehicle battery capacity to charging sessions
charging_with_battery = charging_clean.merge(
    vehicles[["vehicle_id", "battery_kwh"]],
    on="vehicle_id",
    how="left",
    validate="many_to_one"
)

# Identify sessions exceeding battery capacity
battery_issues = charging_with_battery[
    charging_with_battery["kwh"] > charging_with_battery["battery_kwh"]
].copy()

print("Sessions exceeding battery capacity:", len(battery_issues))
print("Sessions missing battery capacity:", charging_with_battery["battery_kwh"].isna().sum())

Sessions exceeding battery capacity: 20
Sessions missing battery capacity: 0


In [18]:
battery_issues.to_csv(project_data / "battery_capacity_issues.csv",
    index=False
)

# 11. Separate Home and Public Charging Sessions

The cleaned charging sessions are divided into two datasets:

- **Home charging:** Sessions identified as home charging.
- **Public charging:** Sessions identified as public charging.

The number of sessions in each group is reported.

The notebook also checks that the combined number of home and public sessions equals the total number of cleaned sessions.

This separation is important for the research questions, especially the investigation of overnight home charging and its relationship to electricity demand and carbon emissions.

In [19]:
# Separate home charging
home_charging = charging_clean[
    charging_clean["home_or_public"] == "home"
].copy()

# Separate public charging
public_charging = charging_clean[
    charging_clean["home_or_public"] == "public"
].copy()

print("Home charging sessions:", len(home_charging))
print("Public charging sessions:", len(public_charging))

print("Total separated sessions:",
      len(home_charging) + len(public_charging))

print("Total cleaned sessions:", len(charging_clean))

Home charging sessions: 2447994
Public charging sessions: 420771
Total separated sessions: 2868765
Total cleaned sessions: 2868765


# 12. Integrate Vehicle and Station Information

Additional information is added to the home and public charging datasets using their corresponding identifiers.

For both datasets, vehicle information is merged using `vehicle_id`.

For public charging sessions, station information is also merged using `station_id`.

The merges use a many-to-one validation rule. This checks that multiple charging sessions can connect to one vehicle or station without creating unintended duplicate rows from the reference datasets.

The resulting datasets contain charging information along with relevant vehicle details. Public charging sessions also contain public station details.

This integration makes it possible to examine charging patterns by vehicle characteristics, charging location, and public station information.

In [20]:
# Add vehicle information to home sessions
home_charging = home_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

# Add vehicle and public station information
public_charging = public_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

public_charging = public_charging.merge(
    stations_clean,
    on="station_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_station")
)

print("Home charging rows:", len(home_charging))
print("Public charging rows:", len(public_charging))

Home charging rows: 2447994
Public charging rows: 420771


# 13. Save the Cleaned Datasets

The following files are created:

- `charging_sessions_clean.csv`: All cleaned and matched charging sessions.
- `home_charging_clean.csv`: Cleaned home charging sessions with vehicle information.
- `public_charging_clean.csv`: Cleaned public charging sessions with vehicle and station information.
- `vehicles_clean.csv`: Cleaned vehicle dataset.
- `stations_clean.csv`: Cleaned public station dataset.

Additional exception files are also saved:
- `sessions_missing_energy.csv`: Sessions with missing energy measurements.
- `battery_capacity_issues.csv`: Sessions with energy greater than the listed vehicle battery capacity.

These exception files preserve information that may be useful when investigating unusual patterns in the data.

In [21]:
charging_clean.to_csv(project_data / "charging_sessions_clean.csv", index=False)

home_charging.to_csv(project_data / "home_charging_clean.csv", index=False)

public_charging.to_csv(project_data / "public_charging_clean.csv", index=False)

vehicles_clean.to_csv(project_data / "vehicles_clean.csv", index=False)

stations_clean.to_csv(project_data / "stations_clean.csv", index=False)

print("All cleaned datasets saved!")

All cleaned datasets saved!


# 14. Review Cleaning Results and Final Row Counts

The final section summarizes the results of the cleaning process.

The cleaning log reports the number of records removed during each operation and the reason for each removal.

The data-quality log reports invalid conversions and other affected values that did not necessarily result in row removal.

The notebook also compares the original and final row counts for each dataset and calculates the percentage of rows retained.

Finally, missing values in the cleaned datasets are reported.


In [22]:
cleaning_report = show_cleaning_log()

display(cleaning_report)

,Dataset,Cleaning Step,Rows Before,Rows After,Rows Removed,Reason
0,Vehicles,Remove missing vehicle IDs,10000,10000,0,A vehicle must have an ID to connect it to cha...
1,Vehicles,Remove duplicate vehicle IDs,10000,10000,0,Each vehicle should appear only once.
2,Stations,Remove missing station IDs,239,239,0,A public station must have an ID to match char...
3,Stations,Remove duplicate station IDs,239,239,0,Each public charging station should appear once.
4,Charging Sessions,Remove missing essential values,2927187,2883142,44045,"Sessions need identifiers, timestamps, electri..."
5,Charging Sessions,Remove duplicate session IDs,2883142,2868795,14347,Each charging session should appear only once.
6,Charging Sessions,Remove invalid energy and location records,2868795,2868765,30,Sessions must have positive energy and a recog...
7,Charging Sessions,Remove sessions without matching vehicles,2868765,2868765,0,Every analyzed charging session must belong to...
8,Charging Sessions,Remove sessions without matching stations,2868765,2868765,0,Public sessions must match stations.csv; home ...


Final before and after report

In [23]:
# FINAL DATASET ROW COUNT COMPARISON

final_counts = {
    "Charging Sessions": len(charging_clean),
    "Vehicles": len(vehicles_clean),
    "Public Stations": len(stations_clean)
}

comparison = pd.DataFrame({
    "Dataset": original_counts.keys(),
    "Original Rows": original_counts.values(),
    "Final Rows": final_counts.values()
})

comparison["Rows Removed"] = (
    comparison["Original Rows"] - comparison["Final Rows"]
)

comparison["Percent Retained"] = (
    comparison["Final Rows"] /
    comparison["Original Rows"] * 100
).round(2)

display(comparison)

,Dataset,Original Rows,Final Rows,Rows Removed,Percent Retained
0,Charging Sessions,2927187,2868765,58422,98.0
1,Vehicles,10000,10000,0,100.0
2,Public Stations,239,239,0,100.0


---

# Part II: Grid and Weather Hourly Data Cleaning

This part incorporates the grid and weather cleaning workflow, preserving the checks, corrections, time-zone alignment, hourly integration, and daily aggregation from `R3-DataCleaning-Grid-Hourly-Corrected.ipynb`. All cleaned outputs are saved alongside the Topic 3 source datasets.


# 15. Load and Inspect Grid and Weather Data

Load both source datasets, inspect the grid columns, and record the original row counts.


In [24]:

import pandas as pd
from pathlib import Path

# Location of the original datasets
data_path = project_data

# Load the datasets
grid = pd.read_csv(data_path / "grid_hourly.csv")
weather = pd.read_csv(data_path / "weather_hourly.csv")

# Record original row counts
print("Original grid rows:", len(grid))
print("Original weather rows:", len(weather))

# Preview both datasets
display(grid.head())
display(weather.head())


Original grid rows: 17520
Original weather rows: 17520


,utc_time_at_end_of_hour,balancing_authority,demand_forecast_mwh,demand_mwh,net_generation,total_interchange,coal,natural_gas,nuclear,wind,solar,hydro,other,unit
0,2022-01-01 07:00,MCPA,634,643,643.0,0.0,284.0,0.0,211.0,106.0,0.0,32.0,10.0,NaN
1,2022-01-01 08:00,MCPA,633,642,642.0,0.0,282.0,0.0,210.0,106.0,0.0,34.0,10.0,NaN
2,2022-01-01 09:00,MCPA,639,641,640.0,0.0,286.0,0.0,209.0,106.0,0.0,30.0,9.0,NaN
3,2022-01-01 10:00,MCPA,638,646,646.0,0.0,302.0,0.0,210.0,94.0,0.0,30.0,10.0,NaN
4,2022-01-01 11:00,MCPA,645,654,654.0,0.0,332.0,0.0,211.0,73.0,0.0,28.0,10.0,NaN


,timestamp,station_id,temp_f,heating_degree_hours,wind_speed_mph,relative_humidity
0,2022-01-01 00:00,WX-0417,40.1,24.9,5,72
1,2022-01-01 01:00,WX-0417,38.1,26.9,4,83
2,2022-01-01 02:00,WX-0417,36.9,28.1,4,88
3,2022-01-01 03:00,WX-0417,35.4,29.6,4,93
4,2022-01-01 04:00,WX-0417,34.8,30.2,4,86


In [25]:
print("Original grid rows:", len(grid))
print("\nColumn names:")
print(grid.columns.tolist())

display(grid.head())

Original grid rows: 17520

Column names:
['utc_time_at_end_of_hour', 'balancing_authority', 'demand_forecast_mwh', 'demand_mwh', 'net_generation', 'total_interchange', 'coal', 'natural_gas', 'nuclear', 'wind', 'solar', 'hydro', 'other', 'unit']


,utc_time_at_end_of_hour,balancing_authority,demand_forecast_mwh,demand_mwh,net_generation,total_interchange,coal,natural_gas,nuclear,wind,solar,hydro,other,unit
0,2022-01-01 07:00,MCPA,634,643,643.0,0.0,284.0,0.0,211.0,106.0,0.0,32.0,10.0,NaN
1,2022-01-01 08:00,MCPA,633,642,642.0,0.0,282.0,0.0,210.0,106.0,0.0,34.0,10.0,NaN
2,2022-01-01 09:00,MCPA,639,641,640.0,0.0,286.0,0.0,209.0,106.0,0.0,30.0,9.0,NaN
3,2022-01-01 10:00,MCPA,638,646,646.0,0.0,302.0,0.0,210.0,94.0,0.0,30.0,10.0,NaN
4,2022-01-01 11:00,MCPA,645,654,654.0,0.0,332.0,0.0,211.0,73.0,0.0,28.0,10.0,NaN


# 16. Inspect Grid Data Quality

Check missing values, duplicates, and negative demand values before applying corrections.


In [26]:

# Check for missing values
print("MISSING VALUES:")
print(grid.isnull().sum())

# Check for duplicate rows
print("\nDUPLICATE ROWS:")
print(grid.duplicated().sum())

# Check for negative electricity demand
print("\nNEGATIVE DEMAND VALUES:")
print((grid["demand_mwh"] < 0).sum())

# Check the different units
print("\nUNIT LABELS:")
print(grid["unit"].value_counts(dropna=False))

# Total rows before cleaning
print("\nTOTAL ROWS:")
print(len(grid))


MISSING VALUES:
utc_time_at_end_of_hour        0
balancing_authority            0
demand_forecast_mwh            0
demand_mwh                     0
net_generation                 0
total_interchange              0
coal                           0
natural_gas                    0
nuclear                        0
wind                          72
solar                          0
hydro                          0
other                          0
unit                       13103
dtype: int64

DUPLICATE ROWS:
0

NEGATIVE DEMAND VALUES:
9

UNIT LABELS:
unit
NaN    13103
MWh     2209
GWh     2208
Name: count, dtype: int64

TOTAL ROWS:
17520


In [27]:

# Find all rows with negative electricity demand
negative_demand = grid[grid["demand_mwh"] < 0]

print("Number of negative demand rows:", len(negative_demand))

# Display the affected records
display(negative_demand[
    ["utc_time_at_end_of_hour",
     "demand_mwh",
     "net_generation",
     "unit"]
])


Number of negative demand rows: 9


,utc_time_at_end_of_hour,demand_mwh,net_generation,unit
1314,2022-02-25 01:00,-687,687.0,NaN
1423,2022-03-01 14:00,-849,847.0,NaN
2107,2022-03-30 02:00,-771,771.0,NaN
3231,2022-05-15 22:00,-650,650.0,NaN
8849,2023-01-05 00:00,-1024,1024.0,NaN
9347,2023-01-25 18:00,-974,974.0,NaN
9689,2023-02-09 00:00,-1019,1020.0,NaN
10394,2023-03-10 09:00,-580,580.0,NaN
13834,2023-10-31 17:00,-785,785.0,MWh


In [28]:

# Compare negative demand with forecasts
negative_demand = grid[grid["demand_mwh"] < 0].copy()

negative_demand["absolute_demand"] = (
    negative_demand["demand_mwh"].abs()
)

display(negative_demand[
    ["utc_time_at_end_of_hour",
     "demand_forecast_mwh",
     "demand_mwh",
     "absolute_demand",
     "net_generation"]
])


,utc_time_at_end_of_hour,demand_forecast_mwh,demand_mwh,absolute_demand,net_generation
1314,2022-02-25 01:00,687,-687,687,687.0
1423,2022-03-01 14:00,845,-849,849,847.0
2107,2022-03-30 02:00,769,-771,771,771.0
3231,2022-05-15 22:00,642,-650,650,650.0
8849,2023-01-05 00:00,1024,-1024,1024,1024.0
9347,2023-01-25 18:00,972,-974,974,974.0
9689,2023-02-09 00:00,1025,-1019,1019,1020.0
10394,2023-03-10 09:00,562,-580,580,580.0
13834,2023-10-31 17:00,811,-785,785,785.0


# 17. Correct Grid Demand and Generation Values

Apply the original grid corrections, including wind gaps and generation-unit standardization; verify the result.


In [29]:

# Create a cleaned copy of the grid data
grid_clean = grid.copy()

# Count negative values before correction
negative_count = (grid_clean["demand_mwh"] < 0).sum()

# Correct negative demand values
grid_clean["demand_mwh"] = grid_clean["demand_mwh"].abs()

# Verify the correction
print("Original rows:", len(grid))
print("Negative values corrected:", negative_count)
print("Rows removed:", len(grid) - len(grid_clean))
print("Remaining rows:", len(grid_clean))
print("Negative values remaining:",
      (grid_clean["demand_mwh"] < 0).sum())


Original rows: 17520
Negative values corrected: 9
Rows removed: 0
Remaining rows: 17520
Negative values remaining: 0


In [30]:

# Find rows with missing wind generation
missing_wind = grid_clean[grid_clean["wind"].isna()].copy()

print("Missing wind values:", len(missing_wind))

# Display the first 10 affected records
display(missing_wind[
    ["utc_time_at_end_of_hour",
     "net_generation",
     "coal",
     "natural_gas",
     "nuclear",
     "wind",
     "solar",
     "hydro",
     "other"]
].head(10))


Missing wind values: 72


,utc_time_at_end_of_hour,net_generation,coal,natural_gas,nuclear,wind,solar,hydro,other
2417,2022-04-12 00:00,756.0,236.0,120.0,208.0,NaN,11.0,70.0,10.0
2418,2022-04-12 01:00,790.0,282.0,120.0,209.0,NaN,0.0,63.0,10.0
2419,2022-04-12 02:00,790.0,275.0,120.0,210.0,NaN,0.0,69.0,10.0
2420,2022-04-12 03:00,766.0,251.0,120.0,210.0,NaN,0.0,69.0,10.0
2421,2022-04-12 04:00,693.0,213.0,120.0,211.0,NaN,0.0,33.0,10.0
2422,2022-04-12 05:00,630.0,273.0,0.0,210.0,NaN,0.0,31.0,10.0
2423,2022-04-12 06:00,557.0,201.0,0.0,209.0,NaN,0.0,31.0,10.0
2424,2022-04-12 07:00,561.0,203.0,0.0,210.0,NaN,0.0,32.0,10.0
2425,2022-04-12 08:00,574.0,221.0,0.0,209.0,NaN,0.0,28.0,10.0
2426,2022-04-12 09:00,588.0,229.0,0.0,210.0,NaN,0.0,33.0,10.0


In [31]:

# Generation sources
sources = [
    "coal", "natural_gas", "nuclear",
    "wind", "solar", "hydro", "other"
]

# Check rows where wind is already known
known_wind = grid_clean[grid_clean["wind"].notna()].copy()

# Calculate the sum of generation sources
known_wind["source_total"] = known_wind[sources].sum(axis=1)

# Compare against reported net generation
known_wind["difference"] = (
    known_wind["net_generation"] - known_wind["source_total"]
)

print("Rows checked:", len(known_wind))
print("Exact matches:", (known_wind["difference"] == 0).sum())
print("Within 2 MWh:", (known_wind["difference"].abs() <= 2).sum())
print("Largest difference:", known_wind["difference"].abs().max())

display(known_wind[
    ["net_generation", "source_total", "difference"]
].head(10))


Rows checked: 17448
Exact matches: 16488
Within 2 MWh: 17448
Largest difference: 2.220446049250313e-16


,net_generation,source_total,difference
0,643.0,643.0,0.0
1,642.0,642.0,0.0
2,640.0,640.0,0.0
3,646.0,646.0,0.0
4,654.0,654.0,0.0
5,769.0,769.0,0.0
6,911.0,911.0,0.0
7,911.0,911.0,0.0
8,889.0,889.0,0.0
9,847.0,847.0,0.0


In [32]:

# List generation sources other than wind
other_sources = [
    "coal", "natural_gas", "nuclear",
    "solar", "hydro", "other"
]

# Find rows with missing wind values
missing_wind = grid_clean[grid_clean["wind"].isna()].copy()

# Calculate missing wind generation
missing_wind["calculated_wind"] = (
    missing_wind["net_generation"]
    - missing_wind[other_sources].sum(axis=1)
)

# Check calculated values
print("Missing wind values:", len(missing_wind))
print("Negative calculated values:",
      (missing_wind["calculated_wind"] < 0).sum())
print("Minimum calculated wind:",
      missing_wind["calculated_wind"].min())
print("Maximum calculated wind:",
      missing_wind["calculated_wind"].max())

display(missing_wind[
    ["utc_time_at_end_of_hour", "calculated_wind"]
].head(10))


Missing wind values: 72
Negative calculated values: 0
Minimum calculated wind: 0.0
Maximum calculated wind: 106.0


,utc_time_at_end_of_hour,calculated_wind
2417,2022-04-12 00:00,101.0
2418,2022-04-12 01:00,106.0
2419,2022-04-12 02:00,106.0
2420,2022-04-12 03:00,106.0
2421,2022-04-12 04:00,106.0
2422,2022-04-12 05:00,106.0
2423,2022-04-12 06:00,106.0
2424,2022-04-12 07:00,106.0
2425,2022-04-12 08:00,106.0
2426,2022-04-12 09:00,106.0


In [33]:

# Count missing values before filling
missing_before = grid_clean["wind"].isna().sum()

# Fill missing wind values using calculated values
grid_clean.loc[missing_wind.index, "wind"] = (
    missing_wind["calculated_wind"]
)

# Verify the results
print("Wind values filled:", missing_before)
print("Missing wind values remaining:",
      grid_clean["wind"].isna().sum())
print("Rows remaining:", len(grid_clean))


Wind values filled: 72
Missing wind values remaining: 0
Rows remaining: 17520


In [34]:
# Inspect original generation values by recorded unit (before conversion)
print("ORIGINAL UNIT LABEL COUNTS:")
print(grid["unit"].value_counts(dropna=False).to_string())
print("\nNET GENERATION BY ORIGINAL UNIT:")
display(grid.groupby("unit", dropna=False)["net_generation"].agg(["count", "min", "median", "max"]))


ORIGINAL UNIT LABEL COUNTS:
unit
NaN    13103
MWh     2209
GWh     2208

NET GENERATION BY ORIGINAL UNIT:


,count,min,median,max
unit,,,,
GWh,2208,0.438,0.728,1.176
MWh,2209,447.000,737.000,1196.000
NaN,13103,436.000,734.000,1299.000


In [35]:
# Convert generation and interchange quantities into MWh.
# Data dictionary: unit applies to net_generation, total_interchange, and fuel columns;
# demand_mwh and demand_forecast_mwh are already in MWh.
generation_columns = [
    "net_generation", "total_interchange", "coal", "natural_gas",
    "nuclear", "wind", "solar", "hydro", "other"
]
gwh_mask = grid_clean["unit"].eq("GWh")
missing_units = grid_clean["unit"].isna().sum()
gwh_rows = gwh_mask.sum()

# GWh values are 1/1000 of comparable MWh values in the source data.
grid_clean.loc[gwh_mask, generation_columns] *= 1000
# The blank-unit rows are on the MWh scale; standardize their metadata too.
grid_clean["unit"] = "MWh"

print("GWh rows converted to MWh:", gwh_rows)
print("Blank unit labels standardized to MWh:", missing_units)
print("Remaining missing unit labels:", grid_clean["unit"].isna().sum())
print("Units after conversion:")
print(grid_clean["unit"].value_counts().to_string())
print("\nNet generation after conversion:")
print(grid_clean["net_generation"].agg(["min", "median", "max"]))


GWh rows converted to MWh: 2208
Blank unit labels standardized to MWh: 13103
Remaining missing unit labels: 0
Units after conversion:
unit
MWh    17520

Net generation after conversion:
min        436.0
median     733.0
max       1299.0
Name: net_generation, dtype: float64


In [36]:

# Final grid validation
print("Total rows:", len(grid_clean))

print("\nMissing values:")
print(grid_clean.isna().sum().to_string())

print("\nNegative demand values:",
      (grid_clean["demand_mwh"] < 0).sum())

print("Duplicate timestamps:",
      grid_clean["utc_time_at_end_of_hour"].duplicated().sum())

print("\nDate range:")
print("First:", grid_clean["utc_time_at_end_of_hour"].min())
print("Last:", grid_clean["utc_time_at_end_of_hour"].max())


Total rows: 17520

Missing values:
utc_time_at_end_of_hour    0
balancing_authority        0
demand_forecast_mwh        0
demand_mwh                 0
net_generation             0
total_interchange          0
coal                       0
natural_gas                0
nuclear                    0
wind                       0
solar                      0
hydro                      0
other                      0
unit                       0

Negative demand values: 0
Duplicate timestamps: 0

Date range:
First: 2022-01-01 07:00
Last: 2024-01-01 06:00


# 18. Inspect and Clean Weather Data

Inspect duplicate timestamps, fill missing measurements, and validate the cleaned weather records.


In [37]:

# Check weather dataset for problems
print("Original weather rows:", len(weather))

print("\nMissing values:")
print(weather.isna().sum().to_string())

print("\nDuplicate rows:", weather.duplicated().sum())

print("Duplicate timestamps:",
      weather["timestamp"].duplicated().sum())

print("\nDate range:")
print("First:", weather["timestamp"].min())
print("Last:", weather["timestamp"].max())


Original weather rows: 17520

Missing values:
timestamp                0
station_id               0
temp_f                  43
heating_degree_hours    43
wind_speed_mph           0
relative_humidity        0

Duplicate rows: 0
Duplicate timestamps: 2

Date range:
First: 2022-01-01 00:00
Last: 2023-12-31 23:00


In [38]:

# Find weather records with repeated timestamps
duplicate_times = weather[
    weather["timestamp"].duplicated(keep=False)
].copy()

print("Records with repeated timestamps:", len(duplicate_times))

# Display the affected records
display(duplicate_times.sort_values("timestamp"))


Records with repeated timestamps: 4


,timestamp,station_id,temp_f,heating_degree_hours,wind_speed_mph,relative_humidity
7416,2022-11-06 01:00,WX-0417,47.9,17.1,4,71
7417,2022-11-06 01:00,WX-0417,45.6,19.4,3,96
16152,2023-11-05 01:00,WX-0417,42.9,22.1,8,100
16153,2023-11-05 01:00,WX-0417,42.4,22.6,8,84


In [39]:

# Create a cleaned copy of the weather dataset
weather_clean = weather.copy()

# Check missing temperature and heating degree hours
missing_temp = weather_clean["temp_f"].isna()
missing_heating = weather_clean["heating_degree_hours"].isna()

print("Missing temperatures:", missing_temp.sum())
print("Missing heating degree hours:", missing_heating.sum())

print("Rows missing both:",
      (missing_temp & missing_heating).sum())

print("Rows missing only temperature:",
      (missing_temp & ~missing_heating).sum())

print("Rows missing only heating degree hours:",
      (~missing_temp & missing_heating).sum())

# Display first 10 affected records
display(weather_clean[
    missing_temp | missing_heating
].head(10))


Missing temperatures: 43
Missing heating degree hours: 43
Rows missing both: 43
Rows missing only temperature: 0
Rows missing only heating degree hours: 0


,timestamp,station_id,temp_f,heating_degree_hours,wind_speed_mph,relative_humidity
360,2022-01-16 00:00,WX-0417,NaN,NaN,0,79
370,2022-01-16 10:00,WX-0417,NaN,NaN,5,79
954,2022-02-09 18:00,WX-0417,NaN,NaN,8,65
1726,2022-03-13 23:00,WX-0417,NaN,NaN,6,60
2193,2022-04-02 10:00,WX-0417,NaN,NaN,8,76
2503,2022-04-15 08:00,WX-0417,NaN,NaN,7,85
2790,2022-04-27 07:00,WX-0417,NaN,NaN,15,83
3122,2022-05-11 03:00,WX-0417,NaN,NaN,9,73
3769,2022-06-07 02:00,WX-0417,NaN,NaN,4,66
3991,2022-06-16 08:00,WX-0417,NaN,NaN,12,70


In [40]:

# Identify consecutive groups of missing temperatures
missing = weather_clean["temp_f"].isna()

group_ids = (missing != missing.shift()).cumsum()

missing_groups = (
    weather_clean[missing]
    .groupby(group_ids[missing])
    .size()
)

print("Total missing temperatures:", missing.sum())
print("Number of missing groups:", len(missing_groups))
print("Longest consecutive missing group:", missing_groups.max())

print("\nGroup sizes:")
print(missing_groups.value_counts().sort_index())


Total missing temperatures: 43
Number of missing groups: 43
Longest consecutive missing group: 1

Group sizes:
1    43
Name: count, dtype: int64


In [41]:

# Count missing temperatures before cleaning
missing_before = weather_clean["temp_f"].isna().sum()

# Fill missing temperatures using linear interpolation
weather_clean["temp_f"] = (
    weather_clean["temp_f"].interpolate(
        method="linear",
        limit=1,
        limit_area="inside"
    )
)

# Verify the results
print("Temperatures filled:", missing_before)
print("Missing temperatures remaining:",
      weather_clean["temp_f"].isna().sum())
print("Rows remaining:", len(weather_clean))


Temperatures filled: 43
Missing temperatures remaining: 0
Rows remaining: 17520


In [42]:

# Check the heating degree hour formula
known = weather_clean["heating_degree_hours"].notna()

calculated = (
    65 - weather_clean.loc[known, "temp_f"]
).clip(lower=0)

difference = (
    weather_clean.loc[known, "heating_degree_hours"]
    - calculated
).abs()

print("Existing records checked:", known.sum())
print("Records matching formula:", (difference < 0.01).sum())
print("Largest difference:", difference.max())
print("Missing heating degree hours:",
      weather_clean["heating_degree_hours"].isna().sum())


Existing records checked: 17477
Records matching formula: 17477
Largest difference: 5.689893001203927e-15
Missing heating degree hours: 43


In [43]:

# Identify missing heating degree hours
missing_heating = weather_clean["heating_degree_hours"].isna()

# Calculate missing values from temperature
weather_clean.loc[missing_heating, "heating_degree_hours"] = (
    65 - weather_clean.loc[missing_heating, "temp_f"]
).clip(lower=0)

# Verify the results
print("Heating degree hours filled:", missing_heating.sum())
print("Missing values remaining:",
      weather_clean["heating_degree_hours"].isna().sum())
print("Rows remaining:", len(weather_clean))


Heating degree hours filled: 43
Missing values remaining: 0
Rows remaining: 17520


In [44]:

# Final weather validation
print("Total rows:", len(weather_clean))

print("\nMissing values:")
print(weather_clean.isna().sum().to_string())

print("\nTemperature range:")
print("Minimum:", weather_clean["temp_f"].min())
print("Maximum:", weather_clean["temp_f"].max())

print("\nHumidity range:")
print("Minimum:", weather_clean["relative_humidity"].min())
print("Maximum:", weather_clean["relative_humidity"].max())

print("\nNegative wind speeds:",
      (weather_clean["wind_speed_mph"] < 0).sum())

print("Duplicate timestamps:",
      weather_clean["timestamp"].duplicated().sum())


Total rows: 17520

Missing values:
timestamp               0
station_id              0
temp_f                  0
heating_degree_hours    0
wind_speed_mph          0
relative_humidity       0

Temperature range:
Minimum: 5.1
Maximum: 97.1

Humidity range:
Minimum: 22
Maximum: 100

Negative wind speeds: 0
Duplicate timestamps: 2


# 19. Align Grid and Weather Timestamps

Check daylight-saving transitions and convert local weather timestamps to UTC before matching hour-ending grid timestamps.


In [45]:

# Display grid timestamps around daylight saving time
grid_times = pd.to_datetime(
    grid_clean["utc_time_at_end_of_hour"]
)

weather_times = pd.to_datetime(
    weather_clean["timestamp"]
)

print("GRID - November 6, 2022:")
display(grid_clean.loc[
    grid_times.between("2022-11-06 05:00", "2022-11-06 10:00"),
    ["utc_time_at_end_of_hour", "demand_mwh"]
])

print("\nWEATHER - November 6, 2022:")
display(weather_clean.loc[
    weather_times.between("2022-11-06 00:00", "2022-11-06 04:00"),
    ["timestamp", "temp_f"]
])


GRID - November 6, 2022:


,utc_time_at_end_of_hour,demand_mwh
7414,2022-11-06 05:00,599
7415,2022-11-06 06:00,557
7416,2022-11-06 07:00,553
7417,2022-11-06 08:00,567
7418,2022-11-06 09:00,572
7419,2022-11-06 10:00,580



WEATHER - November 6, 2022:


,timestamp,temp_f
7415,2022-11-06 00:00,47.8
7416,2022-11-06 01:00,47.9
7417,2022-11-06 01:00,45.6
7418,2022-11-06 02:00,44.3
7419,2022-11-06 03:00,43.2
7420,2022-11-06 04:00,42.9


In [46]:

# Convert weather timestamps to UTC
weather_clean["local_time"] = pd.to_datetime(
    weather_clean["timestamp"]
)

weather_clean["weather_utc"] = (
    weather_clean["local_time"]
    .dt.tz_localize(
        "America/Chicago",
        ambiguous="infer",
        nonexistent="NaT"
    )
    .dt.tz_convert("UTC")
)

# Check the converted timestamps
print("Missing UTC timestamps:",
      weather_clean["weather_utc"].isna().sum())

print("Duplicate UTC timestamps:",
      weather_clean["weather_utc"].duplicated().sum())

print("\nConverted daylight saving timestamps:")
display(weather_clean.loc[
    weather_clean["timestamp"].isin([
        "2022-11-06 01:00",
        "2023-11-05 01:00"
    ]),
    ["timestamp", "weather_utc"]
])


Missing UTC timestamps: 0
Duplicate UTC timestamps: 0

Converted daylight saving timestamps:


,timestamp,weather_utc
7416,2022-11-06 01:00,2022-11-06 06:00:00+00:00
7417,2022-11-06 01:00,2022-11-06 07:00:00+00:00
16152,2023-11-05 01:00,2023-11-05 06:00:00+00:00
16153,2023-11-05 01:00,2023-11-05 07:00:00+00:00


In [47]:

# Convert grid timestamps to UTC
grid_clean["grid_utc"] = pd.to_datetime(
    grid_clean["utc_time_at_end_of_hour"],
    utc=True
)

# Check alignment with weather timestamps
grid_hours = set(grid_clean["grid_utc"])
weather_hours = set(weather_clean["weather_utc"])

# Option 1: Match timestamps directly
direct_matches = len(grid_hours & weather_hours)

# Option 2: Shift weather timestamps forward one hour
shifted_weather_hours = {
    t + pd.Timedelta(hours=1)
    for t in weather_hours
}

shifted_matches = len(grid_hours & shifted_weather_hours)

print("Direct timestamp matches:", direct_matches)
print("Matches with weather shifted +1 hour:", shifted_matches)
print("Total grid records:", len(grid_clean))
print("Total weather records:", len(weather_clean))


Direct timestamp matches: 17519
Matches with weather shifted +1 hour: 17520
Total grid records: 17520
Total weather records: 17520


# 20. Merge Hourly Grid and Weather Data

Merge the cleaned datasets by aligned UTC timestamps and validate the one-to-one match.


In [48]:

# Align weather timestamps with grid hour-ending timestamps
weather_clean["matching_utc"] = (
    weather_clean["weather_utc"] + pd.Timedelta(hours=1)
)

# Merge grid and weather datasets
combined_hourly = pd.merge(
    grid_clean,
    weather_clean,
    left_on="grid_utc",
    right_on="matching_utc",
    how="inner",
    validate="one_to_one"
)

# Verify the merge
print("Original grid rows:", len(grid_clean))
print("Original weather rows:", len(weather_clean))
print("Combined hourly rows:", len(combined_hourly))
print("Missing values:", combined_hourly.isna().sum().sum())
print("Duplicate UTC timestamps:",
      combined_hourly["grid_utc"].duplicated().sum())

# Preview combined data
display(combined_hourly[
    ["grid_utc", "demand_mwh", "temp_f",
     "wind", "coal", "natural_gas"]
].head())


Original grid rows: 17520
Original weather rows: 17520
Combined hourly rows: 17520
Missing values: 0
Duplicate UTC timestamps: 0


,grid_utc,demand_mwh,temp_f,wind,coal,natural_gas
0,2022-01-01 07:00:00+00:00,643,40.1,106.0,284.0,0.0
1,2022-01-01 08:00:00+00:00,642,38.1,106.0,282.0,0.0
2,2022-01-01 09:00:00+00:00,641,36.9,106.0,286.0,0.0
3,2022-01-01 10:00:00+00:00,646,35.4,94.0,302.0,0.0
4,2022-01-01 11:00:00+00:00,654,34.8,73.0,332.0,0.0


# 21. Calculate Daily Averages

Aggregate the merged hourly observations by local calendar date and validate the hourly totals.


In [49]:

# Use the weather observation's local calendar date
combined_hourly["date"] = (
    pd.to_datetime(combined_hourly["timestamp"]).dt.date
)

# Columns to average for each day
average_columns = [
    "demand_mwh",
    "demand_forecast_mwh",
    "net_generation",
    "coal",
    "natural_gas",
    "nuclear",
    "wind",
    "solar",
    "hydro",
    "other",
    "temp_f",
    "heating_degree_hours",
    "wind_speed_mph",
    "relative_humidity"
]

# Calculate daily averages and count hourly records
daily_averages = (
    combined_hourly.groupby("date")
    .agg(
        **{col: (col, "mean") for col in average_columns},
        hours_recorded=("grid_utc", "count")
    )
    .reset_index()
)

# Check results
print("Original hourly records:", len(combined_hourly))
print("Daily records:", len(daily_averages))

print("\nHours per day:")
print(daily_averages["hours_recorded"].value_counts().sort_index())

display(daily_averages.head())


Original hourly records: 17520
Daily records: 730

Hours per day:
hours_recorded
23      2
24    726
25      2
Name: count, dtype: int64


,date,demand_mwh,demand_forecast_mwh,net_generation,coal,natural_gas,nuclear,wind,solar,hydro,other,temp_f,heating_degree_hours,wind_speed_mph,relative_humidity,hours_recorded
0,2022-01-01,743.625000,743.958333,743.625000,302.708333,93.333333,210.083333,65.291667,10.458333,51.791667,9.958333,43.691667,21.308333,6.416667,76.625000,24
1,2022-01-02,741.583333,743.333333,741.625000,333.833333,94.583333,209.708333,29.083333,12.083333,52.333333,10.000000,43.258333,21.741667,6.000000,78.333333,24
2,2022-01-03,863.083333,863.875000,862.958333,396.333333,165.291667,210.208333,17.416667,10.625000,53.166667,9.916667,38.054167,26.945833,8.583333,82.166667,24
3,2022-01-04,823.166667,821.541667,823.416667,342.625000,108.000000,210.041667,88.708333,11.916667,52.000000,10.125000,42.395833,22.604167,11.750000,78.250000,24
4,2022-01-05,791.458333,786.625000,791.500000,351.833333,103.208333,210.041667,51.583333,12.250000,52.625000,9.958333,44.670833,20.329167,7.916667,76.916667,24


In [50]:

# Verify daily aggregation
print("Daily records:", len(daily_averages))

print("Total hours accounted for:",
      daily_averages["hours_recorded"].sum())

print("Original hourly records:",
      len(combined_hourly))

print("Missing daily values:",
      daily_averages.isna().sum().sum())

print("Duplicate dates:",
      daily_averages["date"].duplicated().sum())

print("\nDate range:")
print("First:", daily_averages["date"].min())
print("Last:", daily_averages["date"].max())

# Check that the daily averages preserve the overall demand
weighted_demand = (
    (daily_averages["demand_mwh"] *
     daily_averages["hours_recorded"]).sum()
    / daily_averages["hours_recorded"].sum()
)

print("\nOriginal hourly demand average:",
      combined_hourly["demand_mwh"].mean())

print("Weighted daily demand average:",
      weighted_demand)


Daily records: 730
Total hours accounted for: 17520
Original hourly records: 17520
Missing daily values: 0
Duplicate dates: 0

Date range:
First: 2022-01-01
Last: 2023-12-31

Original hourly demand average: 744.8220890410959
Weighted daily demand average: 744.8220890410959


# 22. Save Cleaned Hourly and Daily Datasets

Write the cleaned hourly and daily datasets to the Topic 3 data directory.


In [51]:

# Save cleaned datasets to the project data folder
output_folder = data_path

# Save hourly dataset
combined_hourly.to_csv(
    output_folder / "combined_hourly_clean.csv",
    index=False
)

# Save daily averages
daily_averages.to_csv(
    output_folder / "combined_daily_averages.csv",
    index=False
)

# Confirm saved files
print("Saved combined_hourly_clean.csv")
print("Saved combined_daily_averages.csv")
print("Hourly rows:", len(combined_hourly))
print("Daily rows:", len(daily_averages))


Saved combined_hourly_clean.csv
Saved combined_daily_averages.csv
Hourly rows: 17520
Daily rows: 730


# Part III

## 23. Clean Fuel Emissions Factors

The fuel_factors.csv dataset contains the average carbon emissions produced per megawatt-hour (MWh) for each electricity source.

This dataset contains eight fuel sources. We will check for missing
values, remove rows with blanks in required columns, remove duplicate fuel
sources, and verify that emission factors are numeric and nonnegative.

Rows with missing required information will not be filled with estimated
values because doing so could introduce inaccurate emissions factors.

# 24. Load the dataset

In [52]:
fuel_factors = pd.read_csv("fuel_factors.csv")

print("Original row count:", len(fuel_factors))
display(fuel_factors)

Original row count: 8


,fuel,kg_co2_per_mwh,note
0,coal,1010.0,fleet average of the BA's coal units
1,natural_gas,430.0,fleet average of the BA's gas units (combined ...
2,nuclear,0.0,no stack emissions
3,wind,0.0,no stack emissions
4,solar,0.0,no stack emissions
5,hydro,0.0,no stack emissions
6,other,0.0,"landfill gas and biomass; biogenic, counted as..."
7,imports,520.0,average of the neighbouring balancing authorit...


# 25. Inspect blanks and data types

In [53]:
print("Missing values by column:")
print(fuel_factors.isna().sum())

print("\nBlank text values by column:")
for column in ["fuel", "note"]:
    print(column, (fuel_factors[column].astype("string").str.strip() == "").sum())

print("\nData types:")
print(fuel_factors.dtypes)

Missing values by column:
fuel              0
kg_co2_per_mwh    0
note              0
dtype: int64

Blank text values by column:
fuel 0
note 0

Data types:
fuel                  str
kg_co2_per_mwh    float64
note                  str
dtype: object


# 26. Clean the data set

In [55]:
fuel_factors_clean = fuel_factors.copy()

# Standardize text
fuel_factors_clean["fuel"] = (
    fuel_factors_clean["fuel"].astype("string").str.strip().str.lower()
)

fuel_factors_clean["note"] = (
    fuel_factors_clean["note"].astype("string").str.strip()
)

# Convert emissions factor to numeric
fuel_factors_clean["kg_co2_per_mwh"] = pd.to_numeric(
    fuel_factors_clean["kg_co2_per_mwh"],
    errors="coerce"
)

# Convert blank strings into missing values
fuel_factors_clean = fuel_factors_clean.replace(r"^\s*$", pd.NA, regex=True)

# Remove rows missing required information
before = len(fuel_factors_clean)

fuel_factors_clean = fuel_factors_clean.dropna(
    subset=["fuel", "kg_co2_per_mwh", "note"]
)

print("Rows removed for missing values:", before - len(fuel_factors_clean))

# Remove duplicate fuel sources
before = len(fuel_factors_clean)

fuel_factors_clean = fuel_factors_clean.drop_duplicates(
    subset=["fuel"],
    keep="first"
)

print("Duplicate fuel sources removed:", before - len(fuel_factors_clean))

# Remove negative emissions factors
before = len(fuel_factors_clean)

fuel_factors_clean = fuel_factors_clean[
    fuel_factors_clean["kg_co2_per_mwh"] >= 0
]

print("Rows removed for negative emissions factors:", before - len(fuel_factors_clean))

print("\nMain cleaned dataset row count:", len(fuel_factors_clean))
display(fuel_factors_clean)

Rows removed for missing values: 0
Duplicate fuel sources removed: 0
Rows removed for negative emissions factors: 0

Main cleaned dataset row count: 8


,fuel,kg_co2_per_mwh,note
0,coal,1010.0,fleet average of the BA's coal units
1,natural_gas,430.0,fleet average of the BA's gas units (combined ...
2,nuclear,0.0,no stack emissions
3,wind,0.0,no stack emissions
4,solar,0.0,no stack emissions
5,hydro,0.0,no stack emissions
6,other,0.0,"landfill gas and biomass; biogenic, counted as..."
7,imports,520.0,average of the neighbouring balancing authorit...


# 27. Create seperate dataset for the fuel types that emit CO2.

In [56]:
fuel_factors_emitting = fuel_factors_clean[
    fuel_factors_clean["kg_co2_per_mwh"] > 0
].copy()

zero_emission_sources = fuel_factors_clean[
    fuel_factors_clean["kg_co2_per_mwh"] == 0
]

print("Positive-emission sources:", len(fuel_factors_emitting))
print("Zero-emission sources:", len(zero_emission_sources))

print("\nZero-emission fuel sources:")
display(zero_emission_sources)

print("\nSources with positive emissions:")
display(fuel_factors_emitting)

Positive-emission sources: 3
Zero-emission sources: 5

Zero-emission fuel sources:


,fuel,kg_co2_per_mwh,note
2,nuclear,0.0,no stack emissions
3,wind,0.0,no stack emissions
4,solar,0.0,no stack emissions
5,hydro,0.0,no stack emissions
6,other,0.0,"landfill gas and biomass; biogenic, counted as..."



Sources with positive emissions:


,fuel,kg_co2_per_mwh,note
0,coal,1010.0,fleet average of the BA's coal units
1,natural_gas,430.0,fleet average of the BA's gas units (combined ...
7,imports,520.0,average of the neighbouring balancing authorit...


# 27. Verify and save the cleaned dataset

In [57]:
print("Main dataset missing values:")
print(fuel_factors_clean.isna().sum())

print("\nMain dataset duplicate fuel sources:",
      fuel_factors_clean["fuel"].duplicated().sum())

print("\nEmitting dataset missing values:")
print(fuel_factors_emitting.isna().sum())

print("\nFinal row counts:")
print("All fuel sources:", len(fuel_factors_clean))
print("Positive-emission sources:", len(fuel_factors_emitting))

# Save both datasets
fuel_factors_clean.to_csv("fuel_factors_clean.csv", index=False)
fuel_factors_emitting.to_csv("fuel_factors_emitting.csv", index=False)

Main dataset missing values:
fuel              0
kg_co2_per_mwh    0
note              0
dtype: int64

Main dataset duplicate fuel sources: 0

Emitting dataset missing values:
fuel              0
kg_co2_per_mwh    0
note              0
dtype: int64

Final row counts:
All fuel sources: 8
Positive-emission sources: 3
